# Build the variant index (and embed the 9 fever rows)

Two jobs in one GPU session, because both need bge-m3 loaded and loading it is
most of the cost.

**Part A - 9 fever rows.** ids 2001-2009 are in the KB but have no vectors, so
retrieval cannot reach them. Nobody asking about fever in pregnancy gets them
today.

**Part B - the variant index.** 3,910 colloquial rewordings of the 2,000 KB
questions. This is the one unblocked item that moves two failing targets:

| | target | now |
|---|---:|---:|
| Recall@1 | 0.70 | 0.528 |
| Recall@5 | 0.90 | 0.645 |
| Recall@20 | - | 0.762 |

Recall@20 sitting far above Recall@1 is the whole argument. For three-quarters
of gold queries the right row is already retrieved and merely not first - the
KB's questions are FAQ-shaped and real questions are not. A variant gives each
row a surface that matches how the question actually gets asked.

**Part C measures the lift** with the legs off and on, on the same 248 queries,
in the same session. The flag ships **off**; it gets flipped only if Part C
earns it.

Runtime: ~25 min on a T4. Needs `QDRANT_URL` and `QDRANT_API_KEY` in Colab
Secrets (key icon in the left sidebar, both toggled on for this notebook).

In [ ]:
!pip install -q FlagEmbedding qdrant-client pandas 2>/dev/null
print("deps installed")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["QDRANT_URL"] = userdata.get("QDRANT_URL")
    os.environ["QDRANT_API_KEY"] = userdata.get("QDRANT_API_KEY")
    print("credentials loaded from Colab Secrets")
except ImportError:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    os.environ["QDRANT_URL"] = s.get_secret("QDRANT_URL")
    os.environ["QDRANT_API_KEY"] = s.get_secret("QDRANT_API_KEY")
    print("credentials loaded from Kaggle Secrets")

In [ ]:
# Clone the branch rather than upload files, so this cannot run against a
# stale copy of the KB or the variant queue.
!rm -rf naari-ai
!git clone -q --depth 1 -b sana/phase3 https://github.com/sana200420/naari-ai.git
%cd naari-ai
print("cloned")

In [ ]:
import pandas as pd

KB = "knowledge_base/Womens_Health_KB - 2000_final.csv"
kb = pd.read_csv(KB)
fever = kb[kb.id > 2000].copy()

variants = pd.read_csv("data/processed/variant_queue_clean.csv")
usable = variants[variants.review_status == "auto_ok"].copy()

print(f"KB rows            : {len(kb)}")
print(f"fever rows to embed: {len(fever)}  ids {list(fever.id)}")
print(f"variants total     : {len(variants)}")
print(f"  usable (auto_ok) : {len(usable)}")
print(f"  regenerate       : {int((variants.review_status == 'regenerate').sum())}")

assert len(fever) == 9, "expected 9 fever rows"
assert usable.source_kb_id.isin(kb.id).all(), "a variant points at an unknown KB id" 

In [ ]:
from FlagEmbedding import BGEM3FlagModel
import torch

print("cuda:", torch.cuda.is_available())
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=torch.cuda.is_available())
print("model loaded")

In [ ]:
from retrieval.normalize import normalize_sd

def embed(texts, batch_size=32):
    """normalize_sd is the ONLY way text reaches a model anywhere in this
    project. Embedding a raw string here would produce vectors the live query
    path can never match, because search normalises every incoming query."""
    normed = [normalize_sd(t) for t in texts]
    out = model.encode(normed, batch_size=batch_size, return_dense=True,
                       return_sparse=True, return_colbert_vecs=False)
    return out["dense_vecs"], out["lexical_weights"]

_d, _s = embed(["test"])
print(f"embed() works, dense dim {len(_d[0])}")

## Part A - the 9 fever rows

New points, so `upsert` rather than `update_vectors`. Point id = KB id, the
convention the collection already uses for canonical rows.

In [ ]:
from qdrant_client import QdrantClient, models

COLLECTION = "naari_ai_kb"
client = QdrantClient(url=os.environ["QDRANT_URL"],
                      api_key=os.environ["QDRANT_API_KEY"], timeout=120)

before = client.count(collection_name=COLLECTION, exact=True).count
print(f"points in collection before: {before}")

# These ids must NOT already exist. If they do, something else wrote them and
# upserting would silently overwrite it.
existing = client.retrieve(collection_name=COLLECTION,
                           ids=[int(i) for i in fever.id],
                           with_payload=False, with_vectors=False)
print(f"ids 2001-2009 already present: {len(existing)}")
assert len(existing) == 0, "these ids already exist - stop and look before writing" 

In [ ]:
def to_sparse(w):
    return models.SparseVector(indices=[int(k) for k in w.keys()],
                               values=[float(v) for v in w.values()])

f_dense, f_sparse = embed(list(fever.question.astype(str)))

client.upsert(
    collection_name=COLLECTION,
    points=[
        models.PointStruct(
            id=int(r.id),
            vector={"dense": d.tolist(), "sparse": to_sparse(s)},
            payload={
                "answer_id": int(r.id),
                "category": r.category,
                "sub_category": r.sub_category,
                "question": r.question,
                "answer": r.answer,
                "source": r.source,
                "review_tier": r.review_tier,
                "lang": "sd",
            },
        )
        for (_, r), d, s in zip(fever.iterrows(), f_dense, f_sparse)
    ],
    wait=True,
)
print(f"upserted {len(fever)} fever rows")

## Part B - the variant index

Variant points go in the **same collection**, separated only by `lang="sd_var"`.
Every existing search filters `lang="sd"`, so variants are invisible to the
current paths until something opts in - the live pipeline cannot regress from
this write alone.

Two details that matter:

**The payload is the canonical row's, not the variant's.** The variant text is
what gets *embedded*; what gets *returned* is the vetted question and answer. A
variant is a way of finding a row, never a thing to show someone.
`variant_question` is kept alongside purely so a result can be traced back to
the phrasing that matched it.

**Point ids are offset by 1,000,000** so they cannot collide with KB ids now or
after the corpus grows.

In [ ]:
VARIANT_ID_OFFSET = 1_000_000

kb_by_id = kb.set_index("id")
usable = usable.reset_index(drop=True)

v_dense, v_sparse = embed(list(usable.variant_question.astype(str)), batch_size=64)
print(f"embedded {len(v_dense)} variants")

In [ ]:
points = []
for i, (_, v) in enumerate(usable.iterrows()):
    src = kb_by_id.loc[v.source_kb_id]
    points.append(models.PointStruct(
        id=VARIANT_ID_OFFSET + i,
        vector={"dense": v_dense[i].tolist(), "sparse": to_sparse(v_sparse[i])},
        payload={
            "answer_id": int(v.source_kb_id),
            "category": src.category,
            "sub_category": src.sub_category,
            "question": src.question,      # canonical, vetted - this is shown
            "answer": src.answer,
            "source": src.source,
            "review_tier": src.review_tier,
            "lang": "sd_var",
            "variant_question": v.variant_question,   # what was embedded
        },
    ))

BATCH = 256
for start in range(0, len(points), BATCH):
    client.upsert(collection_name=COLLECTION,
                  points=points[start:start + BATCH], wait=True)
    print(f"  upserted {min(start + BATCH, len(points))}/{len(points)}", flush=True)
print("variant index written")

In [ ]:
def count_lang(value):
    return client.count(
        collection_name=COLLECTION, exact=True,
        count_filter=models.Filter(must=[models.FieldCondition(
            key="lang", match=models.MatchValue(value=value))])).count

after = client.count(collection_name=COLLECTION, exact=True).count
n_sd, n_var = count_lang("sd"), count_lang("sd_var")
print(f"points before : {before}")
print(f"points after  : {after}   (+{after - before})")
print(f"  lang=sd     : {n_sd}")
print(f"  lang=sd_var : {n_var}")
assert n_sd == len(kb), f"expected {len(kb)} Sindhi rows, found {n_sd}"
assert n_var == len(usable), f"expected {len(usable)} variants, found {n_var}" 

## Part C - measure the lift

The point of the whole exercise. Same 248 gold queries, same session, legs off
then on, so the only thing differing between runs is the variant index.

`VARIANT_WEIGHT` scales how much the variant legs count against the canonical
ones in RRF. 1.0 gives them equal say; lower makes them a tiebreaker rather
than a vote. Sweeping it here is far cheaper than redeploying to find out.

In [ ]:
import time
from retrieval.pipeline import search as retrieval_search

gold = pd.read_csv("eval/gold_eval_280_linked.csv")
print(f"{len(gold)} gold queries")

def score(use_variants, variant_weight=1.0, limit=None):
    g = gold.head(limit) if limit else gold
    ranks, t0 = [], time.time()
    for n, (_, r) in enumerate(g.iterrows(), 1):
        res = retrieval_search(r["query"], top_k=20, candidate_k=20,
                               use_variants=use_variants,
                               variant_weight=variant_weight)
        ids = [x["answer_id"] for x in res["results"]]
        want = int(r.correct_answer_id)
        ranks.append(ids.index(want) + 1 if want in ids else None)
        if n % 50 == 0:
            print(f"    [{n}/{len(g)}] {time.time()-t0:.0f}s", flush=True)
    s = pd.Series(ranks)
    at = lambda k: float((s.notna() & (s <= k)).mean())
    return {"r1": at(1), "r5": at(5), "r20": at(20)}

In [ ]:
# Baseline first, so any later number has something honest to beat.
base = score(use_variants=False)
print(f"OFF   R@1 {base['r1']:.3f}  R@5 {base['r5']:.3f}  R@20 {base['r20']:.3f}")

In [ ]:
results = {"off": base}
for w in [0.5, 1.0]:
    r = score(use_variants=True, variant_weight=w)
    results[f"on_w{w}"] = r
    print(f"ON w={w}  R@1 {r['r1']:.3f}  R@5 {r['r5']:.3f}  R@20 {r['r20']:.3f}"
          f"   (dR@1 {r['r1'] - base['r1']:+.3f})", flush=True)

In [ ]:
rows = []
for name, r in results.items():
    rows.append({"config": name,
                 "recall_at_1": round(r["r1"], 4),
                 "recall_at_5": round(r["r5"], 4),
                 "recall_at_20": round(r["r20"], 4),
                 "delta_r1_vs_off": round(r["r1"] - base["r1"], 4)})
out = pd.DataFrame(rows)
out.to_csv("eval/variant_index_lift.csv", index=False)
print(out.to_string(index=False))

best = max(results.items(), key=lambda kv: kv[1]["r1"])
print(f"best: {best[0]}  R@1 {best[1]['r1']:.3f}  (target 0.70)")
if best[0] == "off":
    print("The variant legs did NOT help. Leave USE_VARIANT_INDEX unset, "
          "and say so in the report.")
else:
    w = best[0].split("_w")[1]
    print(f"Set USE_VARIANT_INDEX=true and VARIANT_WEIGHT={w} in the Space secrets.")

In [ ]:
from google.colab import files
files.download("eval/variant_index_lift.csv")

## Done

Commit `eval/variant_index_lift.csv`. It is the evidence for whether the flag
gets flipped, and it is the number that goes in the report either way.

A null result is still a result worth committing. "Variants did not help,
measured, here is the table" is a finding. Quietly leaving the flag off is not.